
####Objetivo del notebook

Identificar perfiles homogéneos dentro del grupo de víctimas no fatales de violencia intrafamiliar mediante el algoritmo de agrupamiento K-Modes. Grupos que serán utilizados posteriormente para entrenar modelos supervisados de clasificación.

La técnica de agrupamiento K-modes consiste en: 1. Escoger los primeros K centros aleatoriamente, 2. Agrupar los datos de la muestra al centro más cercano, 3. Una vez agrupados, se vuelve a calcular el centro de cada grupo, 4. El punto 2 y 3 se iteran hasta que no se encuentran centros o centroides que representan de mejor manera los grupos. Los centros de los grupos están representados por las modas de cada variable. 

Se aplica la técnica de agrupamiento con 2,3,4 y 5 clusters. y el criterio de decisión se basa fundamentalmente en la medidas costo (disimilitud intra-cluster) y balance, este último representa el % del cluster más pequeño sobre el más grande. 

-Entrada

Base de datos: nofatales_muerte_var_significativas, cuyo origen es el notebook 01_7.

-Salida: 

Tabla : resultados_kmodes

Interpretacion: al final del notebook.

In [0]:
#verificar si kmodes forma parte de las bibliotecas estándar de Databricks
try:
    from kmodes.kmodes import KModes
    print("La librería kmodes está instalada.")
except ImportError:
    print("La librería kmodes NO está instalada.")

In [0]:
%pip install kmodes

In [0]:
%restart_python

In [0]:
#Verificar la instalación

from kmodes.kmodes import KModes

print(KModes)

In [0]:

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import count, lit, col, when, regexp_replace, concat

# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt

# K-Modes
from kmodes.kmodes import KModes

# Tiempo de ejecución
import time

# Ignorar advertencias
import warnings
warnings.filterwarnings("ignore")




In [0]:
#Leer la tabla Delta

tabla = "ml_proyecto_7405607705157039.default.nofatales_muerte_var_significativas"

df = spark.table(tabla)

In [0]:
#Información del DataFrame

print(f"Número de registros: {df.count():,}")

print(f"Número de columnas: {len(df.columns)}")

df.printSchema()

In [0]:
#Separar ambos grupos

df_nofatales = df.filter(col("grupo")=="nofatales")
df_muerte = df.filter(col("grupo")=="muerte")

In [0]:
print("No fatales:", df_nofatales.count())

print("Muerte:", df_muerte.count())

In [0]:
#Variables del estudio

variables = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"

]

In [0]:
# Número de registros
filas = df_nofatales.count()

# Número de variables
columnas = len(variables)

print(f"Filas: {filas:,}")
print(f"Columnas: {columnas}")

In [0]:
#Preparar los datos para K-Modes

#Convertir el data frame de Spark a Pandas

pdf = (
    df_nofatales
    .select(*variables)
    .toPandas()
)

print(pdf.shape)

pdf.head()

In [0]:
#confirmar que todas las columnas son de tipo texto

pdf.dtypes

**Entrenamiento y evaluación de diferentes valores de K**

In [0]:
#Datos para K-Modes

X = pdf.to_numpy()

print(f"Dimensiones: {X.shape}")

In [0]:
#Valores de K a evaluar

valores_k = [2, 3, 4, 5, 6]

In [0]:
#Entrenamiento K-Modes

resultados = []

modelos = {}

for k in valores_k:

    print("=" * 70)
    print(f"Entrenando modelo con K = {k}")

    inicio = time.time()

    modelo = KModes(
        n_clusters=k,
        init="Huang",
        n_init=10,
        max_iter=100,
        random_state=42,
        verbose=0
    )

    # Entrenar modelo
    labels = modelo.fit_predict(X)

    tiempo = time.time() - inicio

    # Tamaño de cada cluster
    cluster_size = (
        pd.Series(labels)
        .value_counts()
        .sort_index()
    )

    # Balance entre el cluster más pequeño y el más grande
    balance = cluster_size.min() / cluster_size.max()

    resultados.append({

        "K": k,

        "Costo": modelo.cost_,

        "Tiempo (seg)": round(tiempo, 2),

        "Iteraciones": getattr(modelo, "n_iter_", np.nan),

        "Cluster más pequeño": int(cluster_size.min()),

        "Cluster más grande": int(cluster_size.max()),

        "Balance": round(balance, 4),

        "Tamaños": cluster_size.tolist()

    })

    # Guardar modelo entrenado
    modelos[k] = modelo

    # Mostrar resumen en pantalla
    print(f"Costo: {modelo.cost_:,.0f}")
    print(f"Tiempo: {tiempo:.2f} segundos")
    print(f"Balance: {balance:.4f}")
    print(f"Tamaños de los clusters: {cluster_size.tolist()}")

print("\nEntrenamiento finalizado.")

In [0]:
#resultados como dataframe
df_resultados = pd.DataFrame(resultados)

In [0]:
#Convertir resultados a Spark DataFrame

df_resultados_spark = spark.createDataFrame(df_resultados)

display(df_resultados_spark)

In [0]:
df_resultados_spark.columns

In [0]:
df_resultados_spark = (
    df_resultados_spark
    .withColumnRenamed("Tiempo (seg)", "tiempo_seg")
    .withColumnRenamed("Cluster más pequeño", "cluster_mas_pequeno")
    .withColumnRenamed("Cluster más grande", "cluster_mas_grande")
)

In [0]:
#Guardar resultados como tabla Delta

df_resultados_spark.write\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable(
        "ml_proyecto_7405607705157039.default.resultados_kmodes"
    )

In [0]:
#verificar que quedó guardada
display(
    spark.table(
        "ml_proyecto_7405607705157039.default.resultados_kmodes"
    )
)

Interpretación: se seleccionaron K= 3 clusters, teniendo en cuenta que la diferencia del costo (disimilitud intra-cluster) es mayor cuando se pasa del cluster 2 al 3. Adicionalmente el balance es de 0.5613, esto quiere decir, que el cluster más pequeño representa 56.13% del más grande, esto garantiza que no se formen agrupaciones muy pequeñas, que no contribuya a definir un perfil. Por otra parte, el algoritmo K-Modes convergió en dos iteraciones. Esto indica que, tras dos ciclos de asignación de los individuos y la actualización de los centroide de los clusters, la composición de los grupos se estabilizó y no se observaron mejoras adicionales en la función de costo. En consecuencia, los tres centroides obtenidos representan la solución estable alcanzada por el algoritmo para esa ejecución.